# 🔧 사전 설정 (Pre-Setup)

**수업 전날까지 이 노트북을 완료하세요.**

설정 내용:
1. 패키지 설치 (langchain-core 1.x.x 기준)
2. Arize Phoenix 서버 연결 (`phoenix serve` 명령어 사용)
3. Prompt Hub 프롬프트 3종 등록
4. 평가 데이터셋 20문항 업로드

In [1]:
# 필요 패키지 설치 — langchain-core 1.x.x 기준
%pip install -q \
    "langchain-core>=1.0,<2.0" \
    "langchain-anthropic>=0.3" \
    "langgraph>=0.2" \
    "langserve[all]>=0.3" \
    "arize-phoenix[evals]>=4.0" \
    openinference-instrumentation-langchain \
    sympy \
    numpy \
    pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ── 프록시 설정 로드 ──────────────────────────────────────────────────────────
# 프록시 ON/OFF: .env 파일에서 USE_PROXY=true/false 로 제어합니다.
# phoenix.evals.LLM 생성 시에는 make_eval_model() 을 사용하세요.
from proxy_config import make_llm, make_eval_model, proxy_patched_anthropic

[proxy_config] USE_PROXY=True, PROXY_URL=http://70.10.15.10:8080


## 1. Phoenix 서버 실행

터미널(별도 창)에서 아래 명령어를 실행하세요:

```bash
phoenix serve
```

서버가 뜨면 http://localhost:6006 에서 UI를 확인할 수 있습니다.

In [2]:
# Phoenix 서버 연결 확인
import requests
try:
    r = requests.get("http://localhost:6006", timeout=3)
    print("✅ Phoenix 서버 응답 확인 — http://localhost:6006")
except Exception as e:
    print(f"❌ Phoenix 서버에 연결할 수 없습니다: {e}")
    print("터미널에서 'phoenix serve' 명령어를 실행하세요.")

✅ Phoenix 서버 응답 확인 — http://localhost:6006


## 2. LangChain → Phoenix Tracing 연결

In [3]:
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

tracer_provider = register(
    project_name="math-agent",
    endpoint="http://localhost:6006/v1/traces",
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
print("✅ Tracing 연결 완료 — 이후 LangChain 호출이 자동으로 Phoenix에 기록됩니다.")

c:\Users\geonjae.joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenTelemetry Tracing Details
|  Phoenix Project: math-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

✅ Tracing 연결 완료 — 이후 LangChain 호출이 자동으로 Phoenix에 기록됩니다.


## 3. Prompt Hub — 프롬프트 3종 등록

In [4]:
from phoenix.client import Client
from phoenix.client.types import PromptVersion

client = Client()

def register_prompt(name: str, template: str):
    """Phoenix Prompt Hub에 시스템 프롬프트를 등록합니다."""
    try:
        version = PromptVersion(
            [{"role": "system", "content": template}],
            model_name="claude-haiku-4-5-20251001",
            model_provider="ANTHROPIC",
            template_format="NONE",
        )
        client.prompts.create(name=name, version=version)
        print(f"✅ '{name}' 등록 완료")
    except Exception as e:
        print(f"ℹ️  '{name}': {e}")
        print("   Phoenix UI → Prompts 탭에서 수동 등록: http://localhost:6006")

In [5]:
# ── 1. 일반 대화 모드 ──────────────────────────────────
CHAT_SYSTEM = """당신은 친절하고 유능한 AI 어시스턴트입니다.
사용자의 질문에 명확하고 도움이 되는 답변을 제공하세요.
답변은 한국어로 작성하세요.
복잡한 내용은 단계별로 설명하세요."""

register_prompt("chat_system", CHAT_SYSTEM)

✅ 'chat_system' 등록 완료


In [6]:
# ── 2. 계산기 모드 ─────────────────────────────────────
CALCULATOR_SYSTEM = """당신은 수학 계산 전문가입니다.
사용자의 계산 요청을 분석하고 반드시 도구를 사용해 계산하세요.

규칙:
1. 머릿속으로 계산하지 말고 반드시 도구를 호출하세요.
2. 계산 결과는 명확하게 표현하세요.
3. 사용한 도구와 입력값을 간략히 설명하세요."""

register_prompt("calculator_system", CALCULATOR_SYSTEM)

✅ 'calculator_system' 등록 완료


In [7]:
# ── 3. 수학 풀이 모드 (숙제용) ──────────────────────────
MATH_SOLVER_SYSTEM = """당신은 수학 문제 풀이 전문가입니다.

풀이 절차:
1. 문제를 이해하고 풀이 전략을 수립하세요.
2. 단계별 풀이 과정을 명확하게 서술하세요.
3. 수치 계산이 필요하면 반드시 계산 도구를 활용하세요.
4. 최종 답을 명확하게 표시하세요.
5. 사용한 수학 개념을 간략히 설명하세요."""

register_prompt("math_solver_system", MATH_SOLVER_SYSTEM)

✅ 'math_solver_system' 등록 완료


In [8]:
# 등록 확인
print("\n등록된 프롬프트 확인:")
for name in ["chat_system", "calculator_system", "math_solver_system"]:
    try:
        p = client.prompts.get(prompt_identifier=name)
        # _template["messages"][0]["content"] 로 시스템 프롬프트 미리보기
        msgs = p._template.get("messages", [])
        preview = str(msgs[0].get("content", ""))[:60].replace("\n", " ") if msgs else ""
        print(f"  ✅ {name}: '{preview}...'")
    except Exception as e:
        print(f"  ❌ {name}: {e}")


등록된 프롬프트 확인:
  ✅ chat_system: '당신은 친절하고 유능한 AI 어시스턴트입니다. 사용자의 질문에 명확하고 도움이 되는 답변을 제공하세요. 답변...'
  ✅ calculator_system: '당신은 수학 계산 전문가입니다. 사용자의 계산 요청을 분석하고 반드시 도구를 사용해 계산하세요.  규칙: 1...'
  ✅ math_solver_system: '당신은 수학 문제 풀이 전문가입니다.  풀이 절차: 1. 문제를 이해하고 풀이 전략을 수립하세요. 2. 단계...'


## 4. 평가 데이터셋 업로드

In [9]:
import pandas as pd

data = [
    # 사칙연산 8문항
    {"input": "23 곱하기 47 더하기 15를 계산해줘", "expected": "1096", "type": "arithmetic", "expression": "23*47+15"},
    {"input": "(100 빼기 37) 곱하기 4를 2로 나눠줘", "expected": "126.0", "type": "arithmetic", "expression": "(100-37)*4/2"},
    {"input": "2의 10제곱을 계산해줘", "expected": "1024", "type": "arithmetic", "expression": "2**10"},
    {"input": "15를 4로 나눈 나머지는?", "expected": "3", "type": "arithmetic", "expression": "15%4"},
    {"input": "7 더하기 3의 합에 8 빼기 3을 곱해줘", "expected": "50", "type": "arithmetic", "expression": "(7+3)*(8-3)"},
    {"input": "1000을 5 곱하기 4로 나눠줘", "expected": "50.0", "type": "arithmetic", "expression": "1000/(5*4)"},
    {"input": "3의 세제곱 더하기 4의 세제곱", "expected": "91", "type": "arithmetic", "expression": "3**3+4**3"},
    {"input": "256 나누기 16 더하기 9", "expected": "25.0", "type": "arithmetic", "expression": "256/16+9"},
    # 미분/적분 6문항
    {"input": "x의 3제곱 더하기 2x를 x로 미분해줘", "expected": "3*x**2 + 2", "type": "calculus", "expression": "x**3 + 2*x"},
    {"input": "sin(x)를 x에 대해 미분해줘", "expected": "cos(x)", "type": "calculus", "expression": "sin(x)"},
    {"input": "x 제곱 곱하기 exp(x)를 x로 미분해줘", "expected": "x**2*exp(x) + 2*x*exp(x)", "type": "calculus", "expression": "x**2*exp(x)"},
    {"input": "x의 세제곱을 x에 대해 적분해줘", "expected": "x**4/4", "type": "calculus", "expression": "x**3"},
    {"input": "cos(x)를 x에 대해 적분해줘", "expected": "sin(x)", "type": "calculus", "expression": "cos(x)"},
    {"input": "2x 더하기 1을 x에 대해 적분해줘", "expected": "x**2 + x", "type": "calculus", "expression": "2*x + 1"},
    # 행렬 연산 6문항
    {"input": "[[1,2],[3,4]] 행렬의 행렬식을 구해줘", "expected": "-2.0", "type": "matrix", "expression": "det([[1,2],[3,4]])"},
    {"input": "[[2,0],[0,3]] 행렬의 행렬식은?", "expected": "6.0", "type": "matrix", "expression": "det([[2,0],[0,3]])"},
    {"input": "[[1,2],[3,4]] 행렬의 역행렬을 구해줘", "expected": "[[-2.0, 1.0], [1.5, -0.5]]", "type": "matrix", "expression": "inv([[1,2],[3,4]])"},
    {"input": "단위 행렬 [[1,0],[0,1]]의 역행렬은?", "expected": "[[1.0, 0.0], [0.0, 1.0]]", "type": "matrix", "expression": "inv([[1,0],[0,1]])"},
    {"input": "[[1,2],[3,4]] 곱하기 [[5,6],[7,8]] 행렬곱 계산", "expected": "[[19.0, 22.0], [43.0, 50.0]]", "type": "matrix", "expression": "matmul([[1,2],[3,4]],[[5,6],[7,8]])"},
    {"input": "[[2,0],[0,2]] 곱하기 [[1,1],[1,1]] 행렬곱", "expected": "[[2.0, 2.0], [2.0, 2.0]]", "type": "matrix", "expression": "matmul([[2,0],[0,2]],[[1,1],[1,1]])"},
]

df = pd.DataFrame(data)
print(f"데이터셋 크기: {len(df)}행")
print(f"유형 분포:")
print(df["type"].value_counts().to_string())
df.head(3)

데이터셋 크기: 20행
유형 분포:
type
arithmetic    8
calculus      6
matrix        6


,input,expected,type,expression
0,23 곱하기 47 더하기 15를 계산해줘,1096,arithmetic,23*47+15
1,(100 빼기 37) 곱하기 4를 2로 나눠줘,126.0,arithmetic,(100-37)*4/2
2,2의 10제곱을 계산해줘,1024,arithmetic,2**10


In [10]:
# Phoenix에 데이터셋 업로드
try:
    dataset = client.datasets.create_dataset(
        name="calculator_eval",
        dataframe=df,
        input_keys=["input"],
        output_keys=["expected"],
        metadata_keys=["type", "expression"],
    )
    print("✅ 데이터셋 'calculator_eval' 업로드 완료")
    print("Phoenix UI → Datasets 탭에서 확인: http://localhost:6006")
except Exception as e:
    print(f"⚠️  업로드 실패: {e}")
    df.to_csv("calculator_eval.csv", index=False, encoding="utf-8-sig")
    print("\n📁 calculator_eval.csv 저장 → Phoenix UI에서 직접 업로드하세요.")

✅ 데이터셋 'calculator_eval' 업로드 완료
Phoenix UI → Datasets 탭에서 확인: http://localhost:6006


## ✅ 사전 설정 완료

| 항목 | 확인 위치 |
|------|-----------|
| Phoenix 서버 | http://localhost:6006 |
| Tracing | Phoenix → Traces 탭 |
| Prompt Hub (3개) | Phoenix → Prompts 탭 |
| 데이터셋 | Phoenix → Datasets 탭 (calculator_eval, 20행) |

> **수업 당일**: `lab.ipynb`를 열고 **§0 확인 셀만 실행**하면 됩니다.
>
> **기본 모델**: `claude-haiku-4-5-20251001` (빠르고 저렴)